In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import os


# 1. WCZYTANIE DANYCH

In [2]:
df = pd.read_csv('2015.csv', encoding='utf-8')

print("=" * 80)
print("ANALIZA STATYSTYCZNA - WORLD HAPPINESS REPORT 2015")
print("=" * 80)
print(f"\nRozmiar próby: {len(df)} krajów")
print(f"Liczba regionów: {df['Region'].nunique()}")
print(f"Regiony: {sorted(df['Region'].unique())}\n")


ANALIZA STATYSTYCZNA - WORLD HAPPINESS REPORT 2015

Rozmiar próby: 158 krajów
Liczba regionów: 10
Regiony: ['Australia and New Zealand', 'Central and Eastern Europe', 'Eastern Asia', 'Latin America and Caribbean', 'Middle East and Northern Africa', 'North America', 'Southeastern Asia', 'Southern Asia', 'Sub-Saharan Africa', 'Western Europe']



# 2. STATYSTYKA OPISOWA (ESTYMACJA PUNKTOWA)

In [3]:
variables = ['Happiness Score', 'Economy (GDP per Capita)', 'Family',
             'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)']

descriptive_stats = df[variables].describe().T
print("\nWskaźniki opisowe:")
print(descriptive_stats[['mean', 'std', 'min', '50%', 'max']].round(4))

print("\n--- SZCZEGÓŁOWE ESTYMATORY PUNKTOWE ---")
for var in variables:
    mean = df[var].mean()
    median = df[var].median()
    std = df[var].std(ddof=1)  # n-1 dla próby
    print(f"\n{var}:")
    print(f"  Średnia (μ):        {mean:.4f}")
    print(f"  Mediana:            {median:.4f}")
    print(f"  Odch. stand. (s):   {std:.4f}")



Wskaźniki opisowe:
                                 mean     std    min     50%     max
Happiness Score                5.3757  1.1450  2.839  5.2325  7.5870
Economy (GDP per Capita)       0.8461  0.4031  0.000  0.9102  1.6904
Family                         0.9910  0.2724  0.000  1.0295  1.4022
Health (Life Expectancy)       0.6303  0.2471  0.000  0.6967  1.0252
Freedom                        0.4286  0.1507  0.000  0.4355  0.6697
Trust (Government Corruption)  0.1434  0.1200  0.000  0.1072  0.5519

--- SZCZEGÓŁOWE ESTYMATORY PUNKTOWE ---

Happiness Score:
  Średnia (μ):        5.3757
  Mediana:            5.2325
  Odch. stand. (s):   1.1450

Economy (GDP per Capita):
  Średnia (μ):        0.8461
  Mediana:            0.9102
  Odch. stand. (s):   0.4031

Family:
  Średnia (μ):        0.9910
  Mediana:            1.0295
  Odch. stand. (s):   0.2724

Health (Life Expectancy):
  Średnia (μ):        0.6303
  Mediana:            0.6967
  Odch. stand. (s):   0.2471

Freedom:
  Średnia (μ):   

# 3. PRZEDZIAŁY UFNOŚCI (95%)

In [4]:
confidence_level = 0.95
alpha = 1 - confidence_level
n = len(df)
t_critical = stats.t.ppf(1 - alpha/2, df=n-1)

print("\nPrzedziały ufności dla średnich (poziom ufności 95%):")
print("Wzór: x̄ ± t(α/2, n-1) * (s / √n)\n")

for var in variables:
    mean = df[var].mean()
    std = df[var].std(ddof=1)
    error = t_critical * (std / np.sqrt(n))
    ci_lower = mean - error
    ci_upper = mean + error

    print(f"{var}:")
    print(f"  Średnia:  {mean:.4f}")
    print(f"  [PU 95%]: [{ci_lower:.4f}; {ci_upper:.4f}]")
    print(f"  Błąd:     ±{error:.4f}\n")



Przedziały ufności dla średnich (poziom ufności 95%):
Wzór: x̄ ± t(α/2, n-1) * (s / √n)

Happiness Score:
  Średnia:  5.3757
  [PU 95%]: [5.1958; 5.5557]
  Błąd:     ±0.1799

Economy (GDP per Capita):
  Średnia:  0.8461
  [PU 95%]: [0.7828; 0.9095]
  Błąd:     ±0.0633

Family:
  Średnia:  0.9910
  [PU 95%]: [0.9482; 1.0338]
  Błąd:     ±0.0428

Health (Life Expectancy):
  Średnia:  0.6303
  [PU 95%]: [0.5914; 0.6691]
  Błąd:     ±0.0388

Freedom:
  Średnia:  0.4286
  [PU 95%]: [0.4049; 0.4523]
  Błąd:     ±0.0237

Trust (Government Corruption):
  Średnia:  0.1434
  [PU 95%]: [0.1246; 0.1623]
  Błąd:     ±0.0189



# 4. ANALIZA WARIANCJI - ANOVA

In [5]:
print("\n--- TEST ANOVA: Szczęście między regionami ---")
print("Hipoteza H0: μ₁ = μ₂ = ... = μₖ (brak różnic między regionami)")
print("Hipoteza H1: Przynajmniej dwa regiony różnią się średnią\n")

print("Statystyka opisowa dla Happiness Score po regionach:\n")
region_stats = df.groupby('Region')['Happiness Score'].agg([
    ('n', 'count'),
    ('Średnia', 'mean'),
    ('Std. Dev', 'std'),
    ('Min', 'min'),
    ('Max', 'max')
]).round(4)
print(region_stats)

groups = [group['Happiness Score'].values for name, group in df.groupby('Region')]
f_stat, p_value_anova = stats.f_oneway(*groups)

print(f"\n--- WYNIKI ANOVA ---")
print(f"Liczba grup (regionów): {len(groups)}")
print(f"Statystyka F:           {f_stat:.3f}")
print(f"p-value:                {p_value_anova:.3e}")
print(f"Poziom istotności:      α = 0.05")

if p_value_anova < 0.05:
    print(f"\n✓ WNIOSEK: Odrzucamy H0.")
    print(f"  Szczęście RÓŻNI się statystycznie między regionami (p < 0.05)")
else:
    print(f"\n✗ WNIOSEK: Nie ma podstaw do odrzucenia H0.")
    print(f"  Brak istotnych różnic między regionami (p >= 0.05)")



--- TEST ANOVA: Szczęście między regionami ---
Hipoteza H0: μ₁ = μ₂ = ... = μₖ (brak różnic między regionami)
Hipoteza H1: Przynajmniej dwa regiony różnią się średnią

Statystyka opisowa dla Happiness Score po regionach:

                                  n  Średnia  Std. Dev    Min    Max
Region                                                              
Australia and New Zealand         2   7.2850    0.0014  7.284  7.286
Central and Eastern Europe       29   5.3329    0.5704  4.218  6.505
Eastern Asia                      6   5.6262    0.5541  4.874  6.298
Latin America and Caribbean      22   6.1447    0.7286  4.518  7.226
Middle East and Northern Africa  20   5.4069    1.1014  3.006  7.278
North America                     2   7.2730    0.2178  7.119  7.427
Southeastern Asia                 9   5.3174    0.9500  3.819  6.798
Southern Asia                     7   4.5809    0.5705  3.575  5.253
Sub-Saharan Africa               40   4.2028    0.6096  2.839  5.477
Western Europe    

# 5. TEST POST-HOC: TUKEY HSD

In [6]:
print("\nPorównania parami między regionami (jeśli ANOVA istotna):\n")

if p_value_anova < 0.05:
    tukey_result = pairwise_tukeyhsd(endog=df['Happiness Score'],
                                     groups=df['Region'],
                                     alpha=0.05)
    print(tukey_result)
    print("\nInterpretacja: reject=True → znacząca różnica między grupami")
else:
    print("ANOVA nie istotna, test Tukey'a nie stosowany.")



Porównania parami między regionami (jeśli ANOVA istotna):

                         Multiple Comparison of Means - Tukey HSD, FWER=0.05                          
             group1                          group2             meandiff p-adj   lower   upper  reject
------------------------------------------------------------------------------------------------------
      Australia and New Zealand      Central and Eastern Europe  -1.9521 0.0161 -3.7019 -0.2022   True
      Australia and New Zealand                    Eastern Asia  -1.6588 0.1728 -3.6131  0.2954  False
      Australia and New Zealand     Latin America and Caribbean  -1.1403 0.5498  -2.908  0.6274  False
      Australia and New Zealand Middle East and Northern Africa  -1.8781 0.0288 -3.6532  -0.103   True
      Australia and New Zealand                   North America   -0.012    1.0 -2.4055  2.3815  False
      Australia and New Zealand               Southeastern Asia  -1.9676 0.0307 -3.8386 -0.0965   True
      Austral

# 6. ANOVA DLA INNYCH ZMIENNYCH

In [7]:
for var in ['Economy (GDP per Capita)', 'Health (Life Expectancy)', 'Freedom']:
    print(f"\n--- ANOVA dla: {var} ---")
    groups_var = [group[var].values for name, group in df.groupby('Region')]
    f_stat_var, p_value_var = stats.f_oneway(*groups_var)

    print(f"Statystyka F: {f_stat_var:.4f}")
    print(f"p-value:      {p_value_var:.6f}")

    if p_value_var < 0.05:
        print(f"✓ Istotne różnice między regionami (p < 0.05)")
    else:
        print(f"✗ Brak istotnych różnic (p >= 0.05)")



--- ANOVA dla: Economy (GDP per Capita) ---
Statystyka F: 29.2172
p-value:      0.000000
✓ Istotne różnice między regionami (p < 0.05)

--- ANOVA dla: Health (Life Expectancy) ---
Statystyka F: 66.0641
p-value:      0.000000
✓ Istotne różnice między regionami (p < 0.05)

--- ANOVA dla: Freedom ---
Statystyka F: 7.7226
p-value:      0.000000
✓ Istotne różnice między regionami (p < 0.05)


# 7. KORELACJE

In [8]:
print("\n--- Korelacja: GDP - Szczęście ---")
corr_gdp, p_corr_gdp = stats.pearsonr(df['Economy (GDP per Capita)'],
                                       df['Happiness Score'])
print(f"r = {corr_gdp:.4f}, p-value = {p_corr_gdp:.6f}")
if p_corr_gdp < 0.05:
    print("✓ Istotna pozytywna korelacja (p < 0.05)")
else:
    print("✗ Brak istotnej korelacji")

print("\n--- Korelacja: Zdrowie - Szczęście ---")
corr_health, p_corr_health = stats.pearsonr(df['Health (Life Expectancy)'],
                                             df['Happiness Score'])
print(f"r = {corr_health:.4f}, p-value = {p_corr_health:.6f}")
if p_corr_health < 0.05:
    print("✓ Istotna pozytywna korelacja (p < 0.05)")
else:
    print("✗ Brak istotnej korelacji")

print("\n--- Korelacja: Freedom - Szczęście ---")
corr_freedom, p_corr_freedom = stats.pearsonr(df['Freedom'],
                                              df['Happiness Score'])
print(f"r = {corr_freedom:.4f}, p-value = {p_corr_freedom:.6f}")
if p_corr_freedom < 0.05:
    print("✓ Istotna pozytywna korelacja (p < 0.05)")
else:
    print("✗ Brak istotnej korelacji")



--- Korelacja: GDP - Szczęście ---
r = 0.7810, p-value = 0.000000
✓ Istotna pozytywna korelacja (p < 0.05)

--- Korelacja: Zdrowie - Szczęście ---
r = 0.7242, p-value = 0.000000
✓ Istotna pozytywna korelacja (p < 0.05)

--- Korelacja: Freedom - Szczęście ---
r = 0.5682, p-value = 0.000000
✓ Istotna pozytywna korelacja (p < 0.05)


# 8. TEST NORMALNOŚCI

In [9]:
jb_stat, jb_p = stats.jarque_bera(df['Happiness Score'].dropna())
print(f"\nZmienna: Happiness Score")
print(f"Jarque-Bera statystyka: {jb_stat:.4f}")
print(f"p-value:                {jb_p:.3e}")

if jb_p < 0.05:
    print("✗ Rozkład NIE jest normalny (Jarque–Bera p < 0.05)")
else:
    print("✓ Brak podstaw do odrzucenia normalności (Jarque–Bera p ≥ 0.05)")



Zmienna: Happiness Score
Jarque-Bera statystyka: 4.3501
p-value:                1.136e-01
✓ Brak podstaw do odrzucenia normalności (Jarque–Bera p ≥ 0.05)


# 9. WIZUALIZACJA

In [10]:
print("\n" + "=" * 80)
print("GENEROWANIE I ZAPIS WYKRESÓW...")
print("=" * 80)

# Folder na wykresy
plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)

# -----------------------------
# 1) Histogram szczęścia
# -----------------------------
plt.figure(figsize=(8, 5))
plt.hist(df['Happiness Score'], bins=20, color='skyblue', edgecolor='black', alpha=0.8)
plt.axvline(df['Happiness Score'].mean(), color='red', linestyle='--', linewidth=2,
            label=f"Średnia: {df['Happiness Score'].mean():.2f}")
plt.title('Rozkład wskaźnika szczęścia')
plt.xlabel('Happiness Score')
plt.ylabel('Liczba krajów')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '01_hist_happiness.png'), dpi=300, bbox_inches='tight')
plt.close()

# -----------------------------
# 2) Boxplot szczęścia po regionach
# -----------------------------
region_order = df.groupby('Region')['Happiness Score'].median().sort_values(ascending=False).index

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df,
    x='Region',
    y='Happiness Score',
    order=region_order,
    hue='Region',
    palette='Set2',
    legend=False
)
plt.title('Szczęście po regionach (ANOVA)')
plt.xlabel('')
plt.ylabel('Happiness Score')
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '02_boxplot_happiness_by_region.png'), dpi=300, bbox_inches='tight')
plt.close()

# -----------------------------
# 3) Średnie szczęścia po regionach + 95% CI
# -----------------------------
region_means = df.groupby('Region')['Happiness Score'].mean().sort_values(ascending=False)
region_std = df.groupby('Region')['Happiness Score'].std()
region_n = df.groupby('Region').size()
region_ci = 1.96 * region_std / np.sqrt(region_n)

plt.figure(figsize=(10, 6))
plt.barh(range(len(region_means)), region_means.values, xerr=region_ci.values,
         color='coral', edgecolor='black', alpha=0.8, capsize=5)
plt.yticks(range(len(region_means)), region_means.index, fontsize=9)
plt.gca().invert_yaxis()
plt.xlabel('Happiness Score')
plt.title('Średnie szczęścia po regionach (95% CI)')
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '03_mean_happiness_by_region_ci95.png'), dpi=300, bbox_inches='tight')
plt.close()

# -----------------------------
# Funkcja: scatter X vs Happiness
# -----------------------------
def save_scatter_vs_happiness(x_col, file_name, color='steelblue'):
    valid = df[[x_col, 'Happiness Score']].dropna()
    r, p = stats.pearsonr(valid[x_col], valid['Happiness Score'])

    plt.figure(figsize=(7.5, 5.5))
    plt.scatter(valid[x_col], valid['Happiness Score'], alpha=0.65, s=45, color=color)

    # Linia trendu
    z = np.polyfit(valid[x_col], valid['Happiness Score'], 1)
    p_line = np.poly1d(z)
    x_sorted = np.sort(valid[x_col].values)
    plt.plot(x_sorted, p_line(x_sorted), 'r--', linewidth=2, alpha=0.85)

    plt.title(f"{x_col} vs Happiness Score\n(r={r:.3f}, p={p:.4g})")
    plt.xlabel(x_col)
    plt.ylabel('Happiness Score')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(plots_dir, file_name), dpi=300, bbox_inches='tight')
    plt.close()

# -----------------------------
# 4..N) Wszystkie wybrane "vs Happiness"
# -----------------------------
save_scatter_vs_happiness('Economy (GDP per Capita)', '04_gdp_vs_happiness.png', color='tab:blue')
save_scatter_vs_happiness('Health (Life Expectancy)', '05_health_vs_happiness.png', color='tab:green')
save_scatter_vs_happiness('Freedom', '06_freedom_vs_happiness.png', color='tab:orange')
save_scatter_vs_happiness('Family', '07_family_vs_happiness.png', color='tab:purple')
save_scatter_vs_happiness('Trust (Government Corruption)', '08_trust_vs_happiness.png', color='tab:brown')

# Opcjonalne (nie użyte w porównaniu)
save_scatter_vs_happiness('Generosity', '09_generosity_vs_happiness.png', color='tab:pink')
save_scatter_vs_happiness('Dystopia Residual', '10_dystopia_vs_happiness.png', color='tab:gray')

# -----------------------------
# 11) Q-Q plot normalności
# -----------------------------
plt.figure(figsize=(7, 5))
stats.probplot(df['Happiness Score'], dist="norm", plot=plt)
plt.title('Q-Q plot dla Happiness Score')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(plots_dir, '11_qqplot_happiness.png'), dpi=300, bbox_inches='tight')
plt.close()

print(f"✓ Zapisano wykresy do folderu: {plots_dir}")
print("✓ Lista plików:")
for f in sorted(os.listdir(plots_dir)):
    print("  -", f)


GENEROWANIE I ZAPIS WYKRESÓW...
✓ Zapisano wykresy do folderu: plots
✓ Lista plików:
  - 01_hist_happiness.png
  - 02_boxplot_happiness_by_region.png
  - 03_mean_happiness_by_region_ci95.png
  - 04_gdp_vs_happiness.png
  - 05_health_vs_happiness.png
  - 06_freedom_vs_happiness.png
  - 07_family_vs_happiness.png
  - 08_trust_vs_happiness.png
  - 09_generosity_vs_happiness.png
  - 10_dystopia_vs_happiness.png
  - 11_qqplot_happiness.png


# 10. PODSUMOWANIE I WNIOSKI

In [11]:
print("\n1. OGÓLNE STATYSTYKI")
print("-" * 80)
print(f"Liczba krajów w analizie: {len(df)}")
print(f"Liczba regionów: {df['Region'].nunique()}")
print(f"\nŚrednie wskaźniki na świecie:")
print(f"  • Szczęście: {df['Happiness Score'].mean():.2f}/10")
print(f"  • GDP per capita: {df['Economy (GDP per Capita)'].mean():.2f}")
print(f"  • Długość życia: {df['Health (Life Expectancy)'].mean():.2f} lat")
print(f"  • Wolność: {df['Freedom'].mean():.2f}")

print("\n2. PRZEDZIAŁY UFNOŚCI (95%)")
print("-" * 80)
mean_happiness = df['Happiness Score'].mean()
error_happiness = t_critical * (df['Happiness Score'].std(ddof=1) / np.sqrt(n))
print(f"Szczęście na świecie:")
print(f"  Średnia: {mean_happiness:.2f}")
print(f"  Możemy być 95% pewni, że średnia szczęścia w populacji")
print(f"  wynosi między {mean_happiness - error_happiness:.2f} a {mean_happiness + error_happiness:.2f}")

print("\n3. RÓŻNICE MIĘDZY REGIONAMI (ANOVA)")
print("-" * 80)
print(f"Test: Czy szczęście różni się między regionami?")
print(f"Statystyka F: {f_stat:.2f}")
print(f"Wynik testu: p-value = {p_value_anova:.6f}")

if p_value_anova < 0.05:
    print(f"\n✓ TAK - Szczęście ISTOTNIE różni się między regionami!")
    print(f"  To znaczy, że ta różnica nie jest przypadkowa (p < 0.05)")
    print(f"\nRanking szczęścia po regionach:")
    region_ranking = df.groupby('Region')['Happiness Score'].mean().sort_values(ascending=False)
    for i, (region, score) in enumerate(region_ranking.items(), 1):
        print(f"  {i}. {region}: {score:.2f}")
else:
    print(f"\n✗ NIE - Brak istotnych różnic między regionami")

print("\n4. ZWIĄZEK MIĘDZY GDP A SZCZĘŚCIEM")
print("-" * 80)
print(f"Korelacja: r = {corr_gdp:.3f}")
print(f"Wynik testu: p-value = {p_corr_gdp:.6f}")

if p_corr_gdp < 0.05:
    print(f"\n✓ TAK - Bogatsze kraje SĄ bardziej szczęśliwe!")
    print(f"  Związek jest dodatni i statystycznie istotny.")
    print(f"  Im wyższy GDP, tym wyższe szczęście.")
else:
    print(f"\n✗ Brak istotnego związku między GDP a szczęściem")

print("\n5. ZWIĄZEK MIĘDZY ZDROWIEM A SZCZĘŚCIEM")
print("-" * 80)
print(f"Korelacja: r = {corr_health:.3f}")
print(f"Wynik testu: p-value = {p_corr_health:.6f}")

if p_corr_health < 0.05:
    print(f"\n✓ TAK - Dłuższa długość życia WIĄŻE się ze szczęściem!")
    print(f"  To ma sens - zdrowsi ludzie są szczęśliwsi.")
else:
    print(f"\n✗ Brak istotnego związku między zdrowiem a szczęściem")

print("\n6. ZWIĄZEK MIĘDZY WOLNOŚCIĄ A SZCZĘŚCIEM")
print("-" * 80)
print(f"Korelacja: r = {corr_freedom:.3f}")
print(f"Wynik testu: p-value = {p_corr_freedom:.6f}")

if p_corr_freedom < 0.05:
    print(f"\n✓ TAK - Wolność ma wpływ na szczęście!")
    print(f"  Ludzie z większą wolnością wyboru są szczęśliwsi.")
else:
    print(f"\n✗ Brak istotnego związku między wolnością a szczęściem")

print("\n7. ROZKŁAD SZCZĘŚCIA - CZY NORMALNY?")
print("-" * 80)
print(f"Test Jarque-Bera:: p-value = {jb_p:.6f}")

if jb_p >= 0.05:
    print(f"✓ Szczęście rozkłada się normalnie")
    print(f"  To dobrze - testy statystyczne są wiarygodne")
else:
    print(f"✗ Szczęście NIE rozkłada się całkowicie normalnie")
    print(f"  Ale analiza ANOVA jest na tyle odporna, że wyniki są wiarygodne")



1. OGÓLNE STATYSTYKI
--------------------------------------------------------------------------------
Liczba krajów w analizie: 158
Liczba regionów: 10

Średnie wskaźniki na świecie:
  • Szczęście: 5.38/10
  • GDP per capita: 0.85
  • Długość życia: 0.63 lat
  • Wolność: 0.43

2. PRZEDZIAŁY UFNOŚCI (95%)
--------------------------------------------------------------------------------
Szczęście na świecie:
  Średnia: 5.38
  Możemy być 95% pewni, że średnia szczęścia w populacji
  wynosi między 5.20 a 5.56

3. RÓŻNICE MIĘDZY REGIONAMI (ANOVA)
--------------------------------------------------------------------------------
Test: Czy szczęście różni się między regionami?
Statystyka F: 24.76
Wynik testu: p-value = 0.000000

✓ TAK - Szczęście ISTOTNIE różni się między regionami!
  To znaczy, że ta różnica nie jest przypadkowa (p < 0.05)

Ranking szczęścia po regionach:
  1. Australia and New Zealand: 7.29
  2. North America: 7.27
  3. Western Europe: 6.69
  4. Latin America and Caribbean: 6